In [ ]:
import os
import importlib.util

# Habana-specific setup below only matters if the Habana SDK is actually
# installed -- inert (and skipped) on GPU-only or CPU-only machines. It must
# happen before any torch / habana import to take effect, so this checks for
# the package on disk (no import, no side effects) rather than importing it
# to ask. The actual HPU -> CUDA -> CPU device selection happens later via
# leakpro.utils.device.get_device(), which runs the same way regardless of
# what (if anything) is set here.
if importlib.util.find_spec("habana_frameworks") is not None:
    # Enables HPU eager mode when using standard PyTorch (not Habana's fork).
    # Override with PT_HPU_LAZY_MODE=1 in your shell environment if you have
    # Habana's lazy-mode torch fork installed.
    if "PT_HPU_LAZY_MODE" not in os.environ:
        os.environ["PT_HPU_LAZY_MODE"] = "0"

    # On a shared multi-HPU machine, each card is exclusive to one process —
    # if this kernel and another job end up on the same card, Habana's
    # runtime can hard-crash the loser (shows up here as "kernel crashed",
    # no traceback). Run `hl-smi` in a terminal first to see which card is
    # actually idle, then pin to it.
    if "HABANA_VISIBLE_MODULES" not in os.environ:
        os.environ["HABANA_VISIBLE_MODULES"] = "1"

# LLM Membership Inference Attack (EZ-MIA)

This runs `leakpro.llm_attacks.mia`, a standalone error-zone MIA attack for causal LLMs. Unlike the other attacks under `examples/mia/`, it does **not** go through `Leakpro(...)`/`audit.yaml`: it trains its own target LLM and a reference model as part of the experiment, so it's called directly via `run_attack(cfg)` instead. See `leakpro/llm_attacks/mia/README.md` for details.

In [ ]:
import sys


def find_project_root(marker='pyproject.toml'):
    current = os.path.abspath(os.getcwd())
    while True:
        if os.path.exists(os.path.join(current, marker)):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            raise RuntimeError(f"Project root (containing {marker}) not found")
        current = parent


project_root = find_project_root()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from leakpro.llm_attacks.mia.attack import run_attack
from leakpro.llm_attacks.mia.config import load_attack_config_from_yaml
from leakpro.utils.device import get_device

print(f"Using device: {get_device()}")

## Run the attack

Uses `train_config.yaml`, sitting next to this notebook — edit it to change dataset, model, or training hyperparameters. It ships sized for a real experiment (20000 train/eval texts, 3 epochs); for a quick pipeline sanity check, temporarily lower `train_total`/`eval_total`/`val_total`/`epochs`/`sequence_length` in the file instead. More example configs (other datasets/models/reference variants) are in `leakpro/llm_attacks/mia/configs/`.

In [ ]:
config_path = os.path.join(os.getcwd(), "train_config.yaml")
cfg = load_attack_config_from_yaml(config_path)

result = run_attack(cfg)
result

`result` reports `auc`, `tpr_at_fpr_0.01`, and `tpr_at_fpr_0.001` for the run. The CLI (`python -m leakpro.llm_attacks.mia --config ...`) additionally appends each run to `leakpro/llm_attacks/mia/results.csv`; calling `run_attack` directly, as done here, does not.